"""
Notebook 15

Model Deployment Pipeline

AI-based Anomaly Detection
for Industrial Water Treatment Systems

Author:
Hamid Saeli

Purpose
-------
Create an industrial inference pipeline
for predicting anomalies on unseen data.
"""

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np


import joblib

In [2]:
PROJECT_ROOT = Path.cwd().parent

MODEL_PATH = PROJECT_ROOT / "models"

DATA_PATH = PROJECT_ROOT / "data"

RESULT_PATH = PROJECT_ROOT / "results"

MODEL_FILE = MODEL_PATH / "best_model.pkl"

In [3]:
model = joblib.load(MODEL_FILE)

print(type(model))

<class 'sklearn.ensemble._forest.RandomForestClassifier'>


In [4]:
NEW_DATA = PROJECT_ROOT / "data" / "processed" / "test_data.csv"

new_df = pd.read_csv(NEW_DATA)

new_df.head()

,FIT101,LIT101,MV101,P101,P102,AIT201,AIT202,AIT203,FIT201,MV201,...,P501,P502,PIT501,PIT502,PIT503,FIT601,P601,P602,P603,Target
0,0.695766,-0.602819,0.335952,0.606750,-0.046608,-0.248022,-0.785361,-0.092905,0.599198,0.321655,...,0.165048,0.0,0.263217,-0.407415,0.291049,-0.099023,0.0,-0.091453,0.0,0
1,-1.565178,1.075201,0.335952,0.606750,-0.046608,-0.248022,-0.579222,-0.145340,0.619757,0.321655,...,0.165048,0.0,0.205477,-0.140592,0.214762,-0.099023,0.0,-0.091453,0.0,0
2,0.571699,-0.503367,0.335952,0.606750,-0.046608,2.225950,-0.204925,-0.122035,0.609123,0.321655,...,0.165048,0.0,0.121798,-0.514144,0.101149,-0.099023,0.0,-0.091453,0.0,0
3,0.690770,-0.803625,0.335952,0.606750,-0.046608,-0.270778,0.999344,-0.385377,0.599789,0.321655,...,0.165048,0.0,0.067406,-0.780966,0.032440,-0.099023,0.0,-0.091453,0.0,0
4,-1.565178,1.775168,-2.880775,-1.648125,-0.046608,-0.276842,1.313974,0.306767,-1.651437,-3.020295,...,-6.058857,0.0,-6.145809,-3.769375,-6.142788,-0.099023,0.0,-0.091453,0.0,1


In [5]:
if "Target" in new_df.columns:
    X_new = new_df.drop(columns="Target")
else:
    X_new = new_df.copy()

In [6]:
prediction = model.predict(X_new)

prediction[:20]

array([0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0],
      dtype=int64)

In [7]:
probability = model.predict_proba(X_new)

probability[:5]

array([[1.00000000e+00, 0.00000000e+00],
       [1.00000000e+00, 0.00000000e+00],
       [9.69237681e-01, 3.07623194e-02],
       [9.69109010e-01, 3.08909896e-02],
       [5.18039657e-08, 9.99999948e-01]])

In [8]:
anomaly_probability = probability[:,1]

In [9]:
def risk_level(p):

    if p < 0.30:
        return "Low"

    elif p < 0.60:
        return "Medium"

    elif p < 0.85:
        return "High"

    return "Critical"

In [10]:
risk = [risk_level(x) for x in anomaly_probability]

In [11]:
deployment_report = X_new.copy()

deployment_report["Prediction"] = prediction

deployment_report["Anomaly Probability"] = anomaly_probability

deployment_report["Risk Level"] = risk

deployment_report.head()

,FIT101,LIT101,MV101,P101,P102,AIT201,AIT202,AIT203,FIT201,MV201,...,PIT501,PIT502,PIT503,FIT601,P601,P602,P603,Prediction,Anomaly Probability,Risk Level
0,0.695766,-0.602819,0.335952,0.606750,-0.046608,-0.248022,-0.785361,-0.092905,0.599198,0.321655,...,0.263217,-0.407415,0.291049,-0.099023,0.0,-0.091453,0.0,0,0.000000,Low
1,-1.565178,1.075201,0.335952,0.606750,-0.046608,-0.248022,-0.579222,-0.145340,0.619757,0.321655,...,0.205477,-0.140592,0.214762,-0.099023,0.0,-0.091453,0.0,0,0.000000,Low
2,0.571699,-0.503367,0.335952,0.606750,-0.046608,2.225950,-0.204925,-0.122035,0.609123,0.321655,...,0.121798,-0.514144,0.101149,-0.099023,0.0,-0.091453,0.0,0,0.030762,Low
3,0.690770,-0.803625,0.335952,0.606750,-0.046608,-0.270778,0.999344,-0.385377,0.599789,0.321655,...,0.067406,-0.780966,0.032440,-0.099023,0.0,-0.091453,0.0,0,0.030891,Low
4,-1.565178,1.775168,-2.880775,-1.648125,-0.046608,-0.276842,1.313974,0.306767,-1.651437,-3.020295,...,-6.145809,-3.769375,-6.142788,-0.099023,0.0,-0.091453,0.0,1,1.000000,Critical


In [12]:
deployment_report["Risk Level"].value_counts()

Risk Level
Low         276437
Critical     10777
Medium         861
High           269
Name: count, dtype: int64

In [13]:
deployment_report["Prediction"].value_counts()

Prediction
0    277196
1     11148
Name: count, dtype: int64

In [14]:
deployment_report["Anomaly Probability"].describe()

count    288344.000000
mean          0.049018
std           0.189711
min           0.000000
25%           0.000000
50%           0.000003
75%           0.007840
max           1.000000
Name: Anomaly Probability, dtype: float64

In [15]:
critical = deployment_report[
    deployment_report["Risk Level"]=="Critical"
]

critical.head()

,FIT101,LIT101,MV101,P101,P102,AIT201,AIT202,AIT203,FIT201,MV201,...,PIT501,PIT502,PIT503,FIT601,P601,P602,P603,Prediction,Anomaly Probability,Risk Level
4,-1.565178,1.775168,-2.880775,-1.648125,-0.046608,-0.276842,1.313974,0.306767,-1.651437,-3.020295,...,-6.145809,-3.769375,-6.142788,-0.099023,0.0,-0.091453,0.0,1,1.000000,Critical
11,-1.565178,1.778652,-2.880775,-1.648125,-0.046608,-0.484649,1.606909,-0.378968,-1.651437,-3.020295,...,-6.138697,-3.769375,-6.135755,-0.099023,0.0,-0.091453,0.0,1,1.000000,Critical
23,0.540891,-0.749147,0.335952,0.606750,-0.046608,-0.420942,0.920683,-0.333524,0.603570,0.321655,...,0.091257,-0.887695,0.062737,-0.099023,0.0,-0.091453,0.0,1,0.960217,Critical
66,-1.565178,1.778335,-2.880775,-1.648125,-0.046608,-0.208585,0.936960,0.598076,-1.651437,-3.020295,...,-6.153759,-3.769375,-6.151986,-0.099023,0.0,-0.091453,0.0,1,1.000000,Critical
122,0.636369,0.570022,0.335952,-1.648125,-0.046608,3.073867,0.055463,-0.483256,-1.651437,-3.020295,...,0.096276,1.727161,0.083835,-0.098181,0.0,-0.091453,0.0,1,0.984006,Critical


In [16]:
deployment_report.sort_values(
    "Anomaly Probability",
    ascending=False
).head(20)

,FIT101,LIT101,MV101,P101,P102,AIT201,AIT202,AIT203,FIT201,MV201,...,PIT501,PIT502,PIT503,FIT601,P601,P602,P603,Prediction,Anomaly Probability,Risk Level
152388,-1.565178,1.783086,-2.880775,-1.648125,-0.046608,-0.484649,1.661159,-0.437812,-1.651437,-3.020295,...,-6.141207,-3.769375,-6.135214,-0.099023,0.0,-0.091453,0.0,1,1.0,Critical
246226,-1.565178,1.784037,-2.880775,-1.648125,-0.046608,-0.484649,1.606909,-0.416257,-1.651437,-3.020295,...,-6.139952,-3.769375,-6.136837,-0.099023,0.0,-0.091453,0.0,1,1.0,Critical
219729,-1.565178,1.779919,-2.880775,-1.648125,-0.046608,-0.484649,1.634038,-0.481509,-1.651437,-3.020295,...,-6.141207,-3.769375,-6.135214,-0.099023,0.0,-0.091453,0.0,1,1.0,Critical
206664,-1.565178,1.778335,-2.880775,-1.648125,-0.046608,-0.484649,1.620470,-0.494909,-1.651437,-3.020295,...,-6.142044,-3.769375,-6.135214,-0.099023,0.0,-0.091453,0.0,1,1.0,Critical
123034,-1.565178,1.783719,-2.880775,-1.648125,-0.046608,-0.484649,1.606909,-0.416257,-1.651437,-3.020295,...,-6.139952,-3.769375,-6.136837,-0.099023,0.0,-0.091453,0.0,1,1.0,Critical
184758,-1.565178,1.778968,-2.880775,-1.648125,-0.046608,-0.484649,1.601483,-0.489081,-1.651437,-3.020295,...,-6.142044,-3.769375,-6.135214,-0.099023,0.0,-0.091453,0.0,1,1.0,Critical
242008,-1.565178,1.769466,-2.880775,-1.648125,-0.046608,-0.484649,1.625895,-0.431987,-1.651437,-3.020295,...,-6.140789,-3.769375,-6.134673,-0.099023,0.0,-0.091453,0.0,1,1.0,Critical
99513,-1.565178,1.784670,-2.880775,-1.648125,-0.046608,-0.484649,1.628613,-0.463447,-1.651437,-3.020295,...,-6.141207,-3.769375,-6.135755,-0.099023,0.0,-0.091453,0.0,1,1.0,Critical
279743,-1.565178,1.778335,-2.880775,-1.648125,-0.046608,-0.484649,1.625895,-0.443055,-1.651437,-3.020295,...,-6.141207,-3.769375,-6.134673,-0.099023,0.0,-0.091453,0.0,1,1.0,Critical
215245,-1.565178,1.772317,-2.880775,-1.648125,-0.046608,-0.484649,1.606909,-0.433735,-1.651437,-3.020295,...,-6.140789,-3.769375,-6.134673,-0.099023,0.0,-0.091453,0.0,1,1.0,Critical


In [17]:
deployment_report.to_csv(

    RESULT_PATH / "deployment_report.csv",

    index=False

)

In [18]:
def run_inference(dataframe):

    X = dataframe.copy()

    prediction = model.predict(X)

    probability = model.predict_proba(X)[:,1]

    risk = [risk_level(x) for x in probability]

    result = X.copy()

    result["Prediction"] = prediction

    result["Probability"] = probability

    result["Risk Level"] = risk

    return result

In [19]:
sample = X_new.sample(10, random_state=42)

result = run_inference(sample)

result

,FIT101,LIT101,MV101,P101,P102,AIT201,AIT202,AIT203,FIT201,MV201,...,PIT501,PIT502,PIT503,FIT601,P601,P602,P603,Prediction,Probability,Risk Level
201504,0.737954,-0.258220,0.335952,-1.648125,-0.046608,-0.308700,1.650307,-0.684840,-1.651437,-3.020295,...,0.096696,-1.101153,0.068147,-0.099023,0.0,-0.091453,0.0,0,0.288909,Low
117688,-1.565178,1.777068,0.335952,-1.648125,-0.046608,-0.248022,-0.053036,2.237549,-1.651437,0.321655,...,0.249410,-0.407415,0.271031,-0.098602,0.0,-0.091453,0.0,0,0.000000,Low
140536,-1.565178,1.777702,0.335952,-1.648125,-0.046608,-0.248022,-0.256458,2.086069,-1.651437,0.321655,...,0.226398,-0.354050,0.253177,-0.098602,0.0,-0.091453,0.0,0,0.000000,Low
125075,0.540614,-0.453641,0.335952,0.606750,-0.046608,3.048082,-0.497860,-0.261863,0.595063,0.321655,...,-2.091530,-3.769375,-1.712899,-0.098181,0.0,-0.091453,0.0,1,0.998916,Critical
1174,-1.565178,0.802815,-2.880775,0.606750,-0.046608,2.858476,-0.286295,-0.110385,0.620938,0.321655,...,0.182468,1.246884,0.171481,-0.098181,0.0,-0.091453,0.0,0,0.020945,Low
80368,-1.565178,1.209493,0.335952,0.606750,-0.046608,-0.248022,-0.389369,-0.146506,0.615740,0.321655,...,0.148158,1.887255,0.143349,-0.098181,0.0,-0.091453,0.0,0,0.027518,Low
156948,-1.565178,0.354330,0.335952,0.606750,-0.046608,-0.248022,-0.731111,-0.007262,0.619520,0.321655,...,0.351918,1.567071,0.390595,-0.097340,0.0,-0.091453,0.0,0,0.000000,Low
143161,0.709643,-0.524271,0.335952,0.606750,-0.046608,2.800832,-0.408355,-0.184375,0.603570,0.321655,...,0.084979,-0.514144,0.064902,-0.099023,0.0,-0.091453,0.0,0,0.015199,Low
55637,0.678002,0.419892,0.335952,-1.648125,-0.046608,-0.910887,2.019178,-0.863121,-1.651437,-3.020295,...,0.118870,-0.514144,0.083835,-0.099023,0.0,-0.091453,0.0,0,0.095776,Low
133567,0.542279,1.292792,0.335952,-1.648125,-0.046608,-1.065601,2.089705,-0.941191,-1.651437,-3.020295,...,0.160709,-1.047789,0.128200,-0.099023,0.0,-0.091453,0.0,0,0.020558,Low


In [20]:
result.to_csv(

    RESULT_PATH/"deployment_example.csv",

    index=False

)

In [21]:
print("="*70)

print("Deployment Pipeline Completed")

print("="*70)

print()

print("Model Loaded Successfully")

print("Inference Completed")

print("Risk Assessment Generated")

print("Deployment Report Saved")

print()

print(RESULT_PATH)

Deployment Pipeline Completed

Model Loaded Successfully
Inference Completed
Risk Assessment Generated
Deployment Report Saved

E:\new\hamid\CV\AI-based Anomaly Detection and Predictive Monitoring for Industrial Water Treatment Systems\results
